# Experimental test 11

    test ('vq',              # llm_model
        'models--cyankiwi--Qwen3-30B-A3B-Instruct-2507-AWQ-4bit',         # model_ver
        4,                # few_shot_n
        50,                # test_n(# of question for test)
        'Y',              # q_src_yn 
        100,                # iteration num
        'sys_prompt10',   # prompt ver
        5,                # self-consistency number
        0.01,             # temperature
        'ver7'            # excel_verion
        )

### prompt변경하여 테스트


In [1]:
import os
import pandas as pd
from config import config as conf
import re
import numpy as np
from sklearn import metrics



In [4]:
def sc_calc_acc_condition_with_temp_with_sc_model(llm_model, model_ver, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'{conf.DATA_PATH}/{conf.ANNO_RESULT}/{model_ver}'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')] 
    print(f'len of opt_file : {len(opt_file)}')

    df = pd.DataFrame()
    gold_df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp['o_result'] = tmp['result'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp = tmp[tmp['o_result'].isin(['1', '0', '2'])]

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            # print(f'size of the dataset : {df_eval.shape[0]}')
            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list, gold_df


In [5]:
#     test ('vq',              # llm_model
#         'models--cyankiwi--Qwen3-30B-A3B-Instruct-2507-AWQ-4bit',         # model_ver
#         4,                # few_shot_n
#         50,                # test_n(# of question for test)
#         'Y',              # q_src_yn 
#         100,                # iteration num
#         'sys_prompt10',   # prompt ver
#         5,                # self-consistency number
#         0.01,             # temperature
#         'ver7'            # excel_verion
#         )


In [16]:
list_, df_ =         sc_calc_acc_condition_with_temp_with_sc_model('vq', 'models--cyankiwi--Qwen3-30B-A3B-Instruct-2507-AWQ-4bit',  4, 50, 'Y', 100, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

len of opt_file : 82
              precision    recall  f1-score   support

           0      0.997     0.913     0.953      1067
           1      0.696     0.975     0.812       957
           2      0.959     0.607     0.743       801

    accuracy                          0.847      2825
   macro avg      0.884     0.832     0.836      2825
weighted avg      0.884     0.847     0.846      2825

vq_result_4_50_Y :  84.70796460176992
[np.float64(83.87096774193549), np.float64(94.73684210526315), np.float64(74.28571428571429), np.float64(86.48648648648648), np.float64(80.0), np.float64(84.375), np.float64(75.0), np.float64(90.0), np.float64(84.21052631578947), np.float64(81.81818181818183), np.float64(76.31578947368422), np.float64(88.23529411764706), np.float64(89.47368421052632), np.float64(80.55555555555556), np.float64(90.9090909090909), np.float64(82.85714285714286), np.float64(83.33333333333334), np.float64(79.41176470588235), np.float64(90.9090909090909), np.float64(75.0), np.f

In [ ]:
# no boundary pooling
list_, df_ =         sc_calc_acc_condition_with_temp_with_sc_model('vq', 'models--cyankiwi--Qwen3-30B-A3B-Instruct-2507-AWQ-4bit',  4, 50, 'Y', 10, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

len of opt_file : 7
              precision    recall  f1-score   support

           0      0.988     0.924     0.955        92
           1      0.808     0.981     0.886       107
           2      0.973     0.667     0.791        54

    accuracy                          0.893       253
   macro avg      0.923     0.857     0.877       253
weighted avg      0.909     0.893     0.891       253

vq_result_4_50_Y :  89.32806324110672
[np.float64(83.87096774193549), np.float64(86.48648648648648), np.float64(85.29411764705883), np.float64(91.66666666666666), np.float64(90.0), np.float64(92.10526315789474), np.float64(94.5945945945946)]
